In [1]:
import os
# 다중 GPU 충돌 방지 0번 GPU 1개만 사용하도록 고정
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [2]:
import pandas as pd
import json
import torch
import math
import os
from sentence_transformers import InputExample
from sentence_transformers.cross_encoder import CrossEncoder
from torch.utils.data import DataLoader

# 코퍼스 및 분석 데이터 로드
corpus = pd.read_parquet('../data/insk_corpus.parquet')
analyses = pd.read_parquet('../data/article_analyses.parquet')

# 문서 텍스트 재조립 (제목 + 5줄 요약)
merged_df = corpus.merge(analyses, on='article_id', how='left')
merged_df['document_text'] = merged_df['title'].fillna('') + " " + merged_df['summary'].fillna('')

# ID 문자열(str) 강제 변환 및 딕셔너리 매핑
merged_df['article_id_str'] = merged_df['article_id'].astype(str).str.replace(r'\.0$', '', regex=True)
doc_dict = dict(zip(merged_df['article_id_str'], merged_df['document_text']))

print(f"문서 재조립 완료: 총 {len(doc_dict)}건")

문서 재조립 완료: 총 406건


In [3]:
# Triplet 데이터셋 구축
train_triplets = []
jsonl_path = '../data/retrieval_top10_for_reranker.jsonl'

with open(jsonl_path, 'r', encoding='utf-8') as f:
    for line in f:
        item = json.loads(line)
        
        # 정답이 없는 Negative 질문은 학습에서 제외
        if item.get('type') == 'Negative':
            continue
            
        query = item['question']
        gold_ids = [str(gid) for gid in item.get('gold_articles', [])]
        hard_negative_ids = [str(hn_id) for hn_id in item.get('hard_negatives', [])]
        
        # 기사 번호를 실제 텍스트로 매핑
        positive_texts = [doc_dict[gid] for gid in gold_ids if gid in doc_dict]
        negative_texts = [doc_dict[hn_id] for hn_id in hard_negative_ids if hn_id in doc_dict]
        
        # 1:1 매칭 쌍 구성
        for pos_text in positive_texts:
            for neg_text in negative_texts:
                train_triplets.append({
                    'query': query,
                    'positive': pos_text,
                    'negative': neg_text
                })

print(f"파인튜닝용 Triplet 조립 완료: 총 {len(train_triplets)}쌍")

파인튜닝용 Triplet 조립 완료: 총 486쌍


In [4]:
# Cross-encoder 포맷 변환 (Positive: 1.0, Negative: 0.0)
train_examples = []
for item in train_triplets:
    train_examples.append(InputExample(texts=[item['query'], item['positive']], label=1.0))
    train_examples.append(InputExample(texts=[item['query'], item['negative']], label=0.0))

# DataLoader 생성 (CUDA 메모리 부족 시 batch_size 축소)
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=8)

print(f"데이터로더 준비 완료: 총 {len(train_examples)}개 샘플")

데이터로더 준비 완료: 총 972개 샘플


In [5]:
torch.cuda.is_available()

True

In [6]:
import math
import torch
from sentence_transformers.cross_encoder import CrossEncoder

# 사전 학습된 한국어 모델 로드
print("모델을 불러옵니다...")
model = CrossEncoder('klue/bert-base', num_labels=1, max_length=512)

# 학습 하이퍼파라미터 설정
epochs = 3
warmup_steps = math.ceil(len(train_dataloader) * epochs * 0.1) 

# 폴더 구조에 맞춘 저장 경로 (src/reranker/)
save_dir = '../src/reranker/reranker_ft_dir'
os.makedirs('../src/reranker', exist_ok=True)

print("모델 학습을 시작합니다. 잠시만 기다려주세요...")

# 파인튜닝 진행
model.fit(
    train_dataloader=train_dataloader,
    epochs=epochs,
    warmup_steps=warmup_steps,
    output_path=save_dir
)

# 학습 완료된 가중치 저장 (.pt)
save_path = '../src/reranker/reranker_ft.pt'
torch.save(model.model.state_dict(), save_path)
print(f"학습 완료 및 가중치 저장 성공: {save_path}")

모델을 불러옵니다...


config.json:   0%|          | 0.00/425 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

모델 학습을 시작합니다. 잠시만 기다려주세요...


Step,Training Loss


학습 완료 및 가중치 저장 성공: ../src/reranker/reranker_ft.pt


In [8]:
# 1. 뼈대가 되는 베이스 모델 로드
model = CrossEncoder('klue/bert-base', num_labels=1, max_length=512)

# 2. 파인튜닝해서 저장한 가중치(.pt)를 모델에 덮어씌우기
model_path = '../src/reranker/reranker_ft.pt'
model.model.load_state_dict(torch.load(model_path))
model.model.eval() # 모델을 평가(추론) 모드로 고정

print("파인튜닝 가중치 로드 완료")

# 3. 원본 텍스트 재조립 (추론 시 모델에 입력하기 위함)
corpus = pd.read_parquet('../data/insk_corpus.parquet')
analyses = pd.read_parquet('../data/article_analyses.parquet')
merged_df = corpus.merge(analyses, on='article_id', how='left')
merged_df['document_text'] = merged_df['title'].fillna('') + " " + merged_df['summary'].fillna('')
merged_df['article_id_str'] = merged_df['article_id'].astype(str).str.replace(r'\.0$', '', regex=True)
doc_dict = dict(zip(merged_df['article_id_str'], merged_df['document_text']))

# 4. 팀원 A의 1차 검색 결과 로드 (Negative 제외)
qa_data = []
with open('../data/retrieval_top10_for_reranker.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        item = json.loads(line)
        if item.get('type') != 'Negative':
            qa_data.append(item)
            
print(f"총 {len(qa_data)}개 질문")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


파인튜닝 가중치 로드 완료
총 21개 질문


In [13]:
reranked_results = []

for item in qa_data:
    query = item['question']
    q_type = item.get('type', 'Unknown')
    gold_ids = [str(gid) for gid in item.get('gold_articles', [])]
    
    # 1차 검색(Hybrid)에서 가져온 Top 10 기사
    baseline_top10 = [str(rid) for rid in item.get('hybrid_top10', [])]
    
    pairs = []
    valid_ids = []
    
    # Reranker 평가를 위한 문장 쌍 구성
    for rid in baseline_top10:
        if rid in doc_dict:
            pairs.append([query, doc_dict[rid]])
            valid_ids.append(rid)
            
    # CrossEncoder 모델로 10개 문서 점수 다시 계산
    scores = model.predict(pairs)
    
    # 점수 기준 내림차순 정렬
    scored_docs = list(zip(valid_ids, scores))
    scored_docs.sort(key=lambda x: x[1], reverse=True)
    reranked_top10 = [doc[0] for doc in scored_docs]
    
    reranked_results.append({
        'query': query,
        'type': q_type,
        'gold_ids': gold_ids,
        'baseline_top10': baseline_top10,
        'reranked_top10': reranked_top10
    })

print("모든 질문에 대한 Reranking(순위 재조정) 완료!")

모든 질문에 대한 Reranking(순위 재조정) 완료!


In [14]:
def recall_at_k(retrieved_ids, gold_ids, k=5):
    if not gold_ids: return 0.0
    hits = len(set(retrieved_ids[:k]) & set(gold_ids))
    return hits / len(gold_ids)

def mrr_at_k(retrieved_ids, gold_ids, k=5):
    if not gold_ids: return 0.0
    for rank, rid in enumerate(retrieved_ids[:k], 1):
        if rid in gold_ids:
            return 1.0 / rank
    return 0.0

def ndcg_at_k(retrieved_ids, gold_ids, k=5):
    if not gold_ids: return 0.0
    dcg = sum(1.0 / np.log2(rank + 1) for rank, rid in enumerate(retrieved_ids[:k], 1) if rid in gold_ids)
    ideal_hits = min(len(gold_ids), k)
    idcg = sum(1.0 / np.log2(rank + 1) for rank in range(1, ideal_hits + 1))
    return dcg / idcg if idcg > 0 else 0.0

def get_metrics(results, target_key, k=5):
    metrics = {'Strict': {'recall': [], 'mrr': [], 'ndcg': []},
               'Trend':  {'recall': [], 'mrr': [], 'ndcg': []},
               '전체':   {'recall': [], 'mrr': [], 'ndcg': []}}
    
    for res in results:
        q_type = res['type']
        retrieved = res[target_key]
        gold = res['gold_ids']
        
        r = recall_at_k(retrieved, gold, k)
        m = mrr_at_k(retrieved, gold, k)
        n = ndcg_at_k(retrieved, gold, k)
        
        if q_type in metrics:
            metrics[q_type]['recall'].append(r)
            metrics[q_type]['mrr'].append(m)
            metrics[q_type]['ndcg'].append(n)
            
        metrics['전체']['recall'].append(r)
        metrics['전체']['mrr'].append(m)
        metrics['전체']['ndcg'].append(n)

    # 평균 계산
    summary = {}
    for key in metrics:
        summary[key] = {
            'Recall@5': np.mean(metrics[key]['recall']) if metrics[key]['recall'] else 0.0,
            'MRR':      np.mean(metrics[key]['mrr']) if metrics[key]['mrr'] else 0.0,
            'nDCG@5':   np.mean(metrics[key]['ndcg']) if metrics[key]['ndcg'] else 0.0,
        }
    return summary

In [17]:
import pandas as pd
import numpy as np
from IPython.display import display

# Baseline(1차)과 Reranker(2차)의 Top 5 성능 각각 측정
baseline_summary = get_metrics(reranked_results, 'baseline_top10', k=5)
reranker_summary = get_metrics(reranked_results, 'reranked_top10', k=5)

# Ceiling (한계점) 계산: 1차 검색(Baseline)이 Top 10 안에 정답을 얼마나 가져왔는가?
ceiling_recalls = [recall_at_k(res['baseline_top10'], res['gold_ids'], k=10) for res in reranked_results]
ceiling_score = np.mean(ceiling_recalls)

# 1. 전체 결과 요약 데이터프레임 생성
b_m = baseline_summary['전체']
r_m = reranker_summary['전체']

summary_df = pd.DataFrame([
    {'검색 단계': '1차 (Baseline)', 'Recall@5': round(b_m['Recall@5'], 3), 'MRR': round(b_m['MRR'], 3), 'nDCG@5': round(b_m['nDCG@5'], 3)},
    {'검색 단계': '2차 (Reranker)', 'Recall@5': round(r_m['Recall@5'], 3), 'MRR': round(r_m['MRR'], 3), 'nDCG@5': round(r_m['nDCG@5'], 3)}
])

# 2. QA 유형별 성능 향상 분석 데이터프레임 생성
type_data = []
for q_type in ['Strict', 'Trend']:
    b_score = baseline_summary[q_type]["Recall@5"]
    r_score = reranker_summary[q_type]["Recall@5"]
    diff = r_score - b_score
    sign = "+" if diff > 0 else ""
    type_data.append({
        'QA 유형': q_type,
        '1차 (Baseline)': round(b_score, 3),
        '2차 (Reranker)': round(r_score, 3),
        '향상도': f'{sign}{diff:.3f}'
    })

type_df = pd.DataFrame(type_data)

# ==========================================
# 출력부 (display 함수 활용)
# ==========================================

print('='*55)
print('최종 결과 요약 (전체 QA 기준) - Reranker 도입 효과')
print('='*55)
display(summary_df) # DataFrame 표 형태로 깔끔하게 출력

print(f'\nImprovement Ceiling (1차 검색 Top-10 정답 포함률): {ceiling_score:.3f}\n')

print('='*55)
print('QA 유형별 Recall@5 비교')
print('='*55)
display(type_df) # DataFrame 표 형태로 깔끔하게 출력

최종 결과 요약 (전체 QA 기준) - Reranker 도입 효과


,검색 단계,Recall@5,MRR,nDCG@5
0,1차 (Baseline),0.425,0.43,0.344
1,2차 (Reranker),0.536,0.81,0.601



Improvement Ceiling (1차 검색 Top-10 정답 포함률): 0.536

QA 유형별 Recall@5 비교


,QA 유형,1차 (Baseline),2차 (Reranker),향상도
0,Strict,0.476,0.690,+0.214
1,Trend,0.399,0.458,+0.060
